# Video-LLaVA Stage 1 Pretraining on Google Colab (Drive-First)

### Architecture
Since Colab's SSD (~80 GB) cannot hold the full datasets (images: 27 GB, videos: 460+ GB),
we use **Google Drive as the primary data store**:

| Component | Location | Why |
|-----------|----------|-----|
| Image dataset (558K images) | Google Drive | Too large for SSD |
| Video dataset (Valley) | Google Drive | Way too large for SSD |
| Annotation JSONs | Local SSD (copied from Drive) | Tiny files, fast reads |
| Model cache | Local SSD | Fast loading |
| Training checkpoints | Local SSD → synced to Drive | Fast writes, persistent backups |

### Datasets Used (OG Video-LLaVA):
- **Image Pretrain**: LLaVA-558K (`llava_image.zip` from HuggingFace)
- **Video Pretrain**: Valley (`valley_2.zip.001`-`012` from HuggingFace)
- **Annotations**: Official zip from Video-LLaVA authors (Google Drive)

## 1. Check GPU Environment

In [ ]:
!nvidia-smi

## 2. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_ROOT = "/content/drive/MyDrive/Video-LLaVA"
os.makedirs(f"{DRIVE_ROOT}/datasets", exist_ok=True)
os.makedirs(f"{DRIVE_ROOT}/checkpoints", exist_ok=True)
print(f"Google Drive workspace ready at: {DRIVE_ROOT}")

## 3. Clone Repository & Install Dependencies

In [ ]:
%cd /content

# Clone the repo (skip if already cloned)
![ ! -d "grad_project" ] && git clone https://github.com/davidrimon2004/grad_project
%cd /content/grad_project

# Install dependencies (uses Colab's pre-installed PyTorch)
!pip install -q --upgrade pip
!pip install -q transformers tokenizers sentencepiece shortuuid accelerate peft bitsandbytes einops einops-exts timm deepspeed huggingface_hub decord gdown
!apt-get install -y -qq p7zip-full
!pip install -e .

## 4. Option A: Quick Demo / Verification (100 Samples)
Tests the full pipeline end-to-end with dummy data in ~2 minutes. Runs entirely on local SSD.

In [ ]:
!python scripts/colab_pretrain_muler.py \
    --action all \
    --demo_samples 100 \
    --drive_root /content/drive/MyDrive/Video-LLaVA \
    --local_scratch_dir /content/data \
    --local_output_dir /content/checkpoints/videollava-7b-pretrain \
    --num_train_epochs 1.0 \
    --save_steps 25

## 5. Option B: Download & Prepare Full Datasets (Run Once)
Downloads all archives from HuggingFace and annotations from Google Drive,
then extracts everything **directly on Google Drive** (persistent across sessions).

⚠️ **First run takes a while** (downloading ~490 GB + extracting on Drive).
Subsequent sessions skip this step entirely since data is already on Drive.

In [ ]:
!python scripts/colab_pretrain_muler.py \
    --action download \
    --drive_root /content/drive/MyDrive/Video-LLaVA \
    --local_scratch_dir /content/data

## 6. Option C: Full Pretraining (Download + Extract + Train)
One-shot command that handles everything: downloads if needed, extracts if needed, then trains.
Data is read directly from Google Drive. Checkpoints are written to local SSD and synced to Drive.

In [ ]:
!python scripts/colab_pretrain_muler.py \
    --action all \
    --drive_root /content/drive/MyDrive/Video-LLaVA \
    --local_scratch_dir /content/data \
    --local_output_dir /content/checkpoints/videollava-7b-pretrain \
    --learning_rate 1e-3 \
    --num_train_epochs 1.0 \
    --save_steps 500 \
    --save_total_limit 2

## 7. Monitor Training with TensorBoard

In [ ]:
%load_ext tensorboard
%tensorboard --logdir /content/checkpoints/videollava-7b-pretrain

## 8. Check Status

In [ ]:
!python scripts/colab_pretrain_muler.py \
    --action status \
    --drive_root /content/drive/MyDrive/Video-LLaVA \
    --local_scratch_dir /content/data